# NB05 - Train the models (repeated CV, resumable)
`cnn3d` (3D CNN from scratch) is the default. Optional extra models via `cfg.train.models`: `cnn2d`, `resnet10` (set `paths.medicalnet_weights` for transfer learning), `foundation`.
* A checkpoint is written to Drive **after every epoch**. If Colab disconnects, simply re-run the cell: finished folds are skipped, the unfinished one resumes.
* Mixed precision (fp16) is used on GPU.
* Select **Runtime > Change runtime type > T4 GPU** first. Expect roughly 1-3 hours for 15 folds of `cnn3d`.

In [ ]:
# ── 1. Environment Detection (Google Colab vs Local Machine) ──────────────
import sys, os, subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── 2. Configure Paths ───────────────────────────────────────────────────
if IN_COLAB:
    # Google Colab paths (on your Google Drive)
    PROJECT  = '/content/drive/MyDrive/oasis_project'      # this unzipped project folder on Drive
    DATA_DIR = '/content/drive/MyDrive/OASIS1'             # folder with the 12 OASIS-1 disc archives (+ clinical csv)
    OUT_DIR  = '/content/drive/MyDrive/OASIS1_outputs'     # results are saved here (survives Colab disconnects)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{PROJECT}/requirements.txt'], check=False)
else:
    # Local machine paths (runs directly on your laptop)
    PROJECT  = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
    DATA_DIR = os.path.join(PROJECT, 'Dataset')
    OUT_DIR  = os.path.join(PROJECT, 'OASIS1_outputs')

# ── 3. install + load ─────────────────────────────────────────────────────
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
from oasis_ad.config import load_config, Paths
from oasis_ad import pipeline as PL, DISCLAIMER
cfg = load_config(os.path.join(PROJECT, 'configs', 'default.yaml'),
                  {'paths.data_dir': DATA_DIR, 'paths.out_dir': OUT_DIR})
paths = Paths(cfg)
import torch, pandas as pd
print(DISCLAIMER)
print('Environment:', 'Google Colab' if IN_COLAB else 'Local Laptop')
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (on Colab switch to Runtime > Change runtime type > T4 GPU)')

In [ ]:
# OPTIONAL quick test: uncomment to train only repeat 0 / fold 0 first
# PL.step5_train(cfg, paths, repeats=[0], folds=[0])

In [ ]:
PL.step5_train(cfg, paths)   # models listed in cfg.train.models

In [ ]:
import glob, pandas as pd
rows = [__import__('json').load(open(f)) for f in sorted(glob.glob(f'{paths.out_dir}/step5_models/*/r*_f*/fold_done.json'))]
pd.DataFrame(rows)[['model','repeat','fold','epochs_run','best_val_nll','temperature','n_params','train_seconds']]